# Cryptocurrency Data Collection

## Objective

The objective of this notebook is to collect and store historical cryptocurrency market data that will be used throughout the market regime detection pipeline.

The collected datasets will serve as the foundation for exploratory data analysis, feature engineering, dimensionality reduction, clustering, and regime identification.

## Roadmap
1. Project Configuration
2. Output Directory Setup
3. Data Collection
4. Download Summary
5. Dataset Verification
6. Conclusion

-------

## Project Configuration

This section loads the global project configuration and libraries.

All parameters such as assets, dates, intervals and storage locations are defined centrally in `config.py` to ensure consistency across the entire project.

In [1]:
# ==========================================
# Import Libraries and Load Configuration
# ==========================================

import pandas as pd
import yfinance as yf
import sys
import os
from pathlib import Path

# Add the project root to the Python path
project_root = os.path.abspath("..")
if project_root not in sys.path:
    sys.path.append(project_root)

from config import ASSETS, START_DATE, END_DATE, INTERVAL, RAW_DATA_PATH

------

## Output Directory Setup

Create the required output directory if it does not already exist.

In [2]:
# ==========================================
# Create Output Directory
# ==========================================

Path(RAW_DATA_PATH).mkdir(
    parents=True,
    exist_ok=True
)

print(
    f"Output directory ready: "
    f"{RAW_DATA_PATH}"
)

Output directory ready: ../data/raw


-----

## Data Collection

Download historical OHLCV data for each configured asset and save the results as CSV files.

In [3]:
# ==========================================
# Download Historical Data
# ==========================================

download_summary = []

for asset in ASSETS:

    print(f"\nDownloading {asset}...")

    data = yf.download(
        asset,
        start=START_DATE,
        end=END_DATE,
        interval=INTERVAL,
        auto_adjust=False,
        progress=False
    )

    if isinstance(
        data.columns,
        pd.MultiIndex
    ):
        data.columns = (
            data.columns
            .get_level_values(0)
        )

    if data.empty:

        print(
            f"WARNING: No data found for {asset}"
        )

        continue

    asset_name = (
        asset.lower()
             .replace("-", "_")
    )

    filename = (
        f"{RAW_DATA_PATH}/"
        f"{asset_name}_{INTERVAL}_raw.csv"
    )

    data.to_csv(filename)

    download_summary.append(
        {
            "Asset": asset,
            "Rows": len(data),
            "Start_Date": data.index.min(),
            "End_Date": data.index.max(),
            "File": filename
        }
    )

    print(
        f"Saved: {filename}"
    )


Saved: ../data/raw/btc_usd_1d_raw.csv


-------

## Download Summary

Review the datasets that were successfully collected.

The table below summarizes:

- Asset
- Number of observations
- Available date range
- Output file

In [4]:
# ==========================================
# Download Summary
# ==========================================

summary_df = pd.DataFrame(
    download_summary
)

summary_df

,Asset,Rows,Start_Date,End_Date,File
0,BTC-USD,3144,2018-01-01,2026-08-10,../data/raw/btc_usd_1d_raw.csv


### Interpretation

The data collection process completed successfully and produced the historical dataset required for the subsequent stages of the project.

The downloaded data provides multiple years of daily market observations for the configured asset(s), supplying sufficient historical information for the exploratory analysis and modeling stages that follow.

The datasets have been stored in the project's raw data directory, preserving the original collected market information for use throughout the pipeline.

The next stages of the project will examine data quality and market characteristics in greater detail before applying feature engineering, dimensionality reduction, clustering, and market regime detection.

-------

## Dataset Verification

Load the saved files and verify that they were written correctly.

This step acts as a quality-control checkpoint before moving to exploratory data analysis.

In [5]:
# ==========================================
# Verify Saved Files
# ==========================================

expected_columns = [
    "Date",
    "Open",
    "High",
    "Low",
    "Close",
    "Adj Close",
    "Volume"
]

for asset in ASSETS:

    asset_name = (
        asset.lower()
             .replace("-", "_")
    )

    filename = (
        f"{RAW_DATA_PATH}/"
        f"{asset_name}_{INTERVAL}_raw.csv"
    )

    try:

        df = pd.read_csv(filename)

        print(
            f"{asset}: "
            f"{df.shape}"
        )

        missing_columns = [
            column
            for column in expected_columns
            if column not in df.columns
        ]

        if missing_columns:
            print(
                f"{asset}: missing columns "
                f"{missing_columns}"
            )
        else:
            print(
                f"{asset}: "
                f"OHLCV structure verified"
            )

    except FileNotFoundError:

        print(
            f"{asset}: file not found"
        )

BTC-USD: (3144, 7)
BTC-USD: OHLCV structure verified


### Interpretation

The verification step confirms that the collected datasets were successfully saved to the expected raw data directory, can be loaded without errors, and contain the expected OHLCV column structure. The reported dimensions also confirm that historical observations were successfully stored for each configured asset.

This provides a basic quality check of the data-collection stage. More detailed analysis of data types, missing values, statistical characteristics, and market behavior will be performed during the exploratory data analysis stage.

The verified datasets are therefore ready to serve as the raw-data inputs for the subsequent stages of the project.

-------

## Conclusion

The data acquisition stage was successfully completed, producing validated historical cryptocurrency market datasets suitable for quantitative analysis.

Quality-control checks confirmed that the downloaded files were stored correctly and preserve the expected OHLCV market structure required throughout the project pipeline.

These datasets provide the historical foundation needed to investigate market behavior, engineer predictive features, reduce dimensionality through PCA, identify market regimes using clustering techniques, and ultimately build a robust market regime detection framework.

The next notebook focuses on exploratory data analysis (EDA), where data quality, statistical characteristics, and market behavior will be examined in greater detail.